# Deney 1 — Eyebrow ROI Swin V2 Tiny Baseline — PDF Standard Compliant

Bu notebook, ekip arkadaşının **Eye ROI Swin V2 Tiny** deneyini model ve eğitim mantığını koruyarak **kaş ROI** verisine uyarlar.

## Veri kaynağı
Notebook yalnızca aşağıdaki kaş ROI verisini tüketir:

`AISC DeepFake Çalışmaları / Deneyler / Nazlıcan / Deney 1 / Kaş`

- ROI metadata: `Kaş/metadata.csv`
- Kaynak video kimliğini doğrulamak için: `Deneyler/Deney 1 Frame/secim_metadata.csv`
- Ham/ROI görüntülerine yazılmaz; veri salt okunur kullanılır.
- `Kaş/metadata.csv` içindeki başarılı (`SUCCESS`/`OK`) ROI kayıtları eğitim girdisidir.
- Kaş metadata içindeki `source_video` boş olduğunda değer **tahmin edilmez**; `secim_metadata.csv` içindeki `orijinal_yol` ile frame adı üzerinden doğrulanmış biçimde geri kazanılır.

## Sonuç hedefi
Her yeni deney şu yapıda yazılır:

`AISC DeepFake Çalışmaları / Deneyler / Nazlıcan / Deney 1 / Sonuçlar / <run_id>/`

```text
<run_id>/
├── checkpoints/
├── logs/
├── metrics/
├── predictions/
├── figures/
├── artifacts/
├── config_resolved.yaml
├── requirements_lock.txt
├── environment.json
├── output_manifest.csv
└── run_summary.json
```

Run ID: `YYYYMMDD_HHMM_eyebrow_swinv2_tiny_seed42`

## Kalite kapıları
Tam eğitim başlamadan önce metadata/schema, veri muhasebesi, kaynak-video split izolasyonu, 2-batch smoke test, NaN/Inf ve checkpoint continuity testleri çalışır. Test kümesi yalnızca nihai değerlendirmede kullanılır; threshold validation setinden seçilir. Grafikler İngilizce ve kısa kenarı en az 600 px olacak şekilde doğrulanır.


In [ ]:
# ============================================================
# 0. OPTIONAL PINNED AUXILIARY DEPENDENCIES
# ============================================================
# Colab torch/torchvision ikilisini CUDA uyumluluğunu bozmamak için değiştirmiyoruz.
# Bu notebook unpinned pip install kullanmaz.
#
# Temiz bir ortamda yardımcı paketleri sabitlemek isterseniz:
INSTALL_PINNED_AUX = False

PINNED_AUX = [
    "pandas==2.2.3",
    "numpy==2.3.5",
    "scikit-learn==1.8.0",
    "matplotlib==3.10.8",
    "Pillow==12.2.0",
    "PyYAML==6.0.3",
    "tqdm==4.67.3",
]

if INSTALL_PINNED_AUX:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *PINNED_AUX
    ])
else:
    print("Pinned auxiliary install skipped; current Colab runtime will be audited and locked.")

In [ ]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# 2. CREATE MODULAR RUNTIME PACKAGE STRUCTURE
# ============================================================
from pathlib import Path
import shutil

WORKSPACE = Path("/content/eyebrow_swinv2_experiment")
SRC_ROOT = WORKSPACE / "src"

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

for p in [
    SRC_ROOT / "deepfake_roi",
    SRC_ROOT / "deepfake_roi" / "data",
    SRC_ROOT / "deepfake_roi" / "models",
    SRC_ROOT / "deepfake_roi" / "training",
    SRC_ROOT / "deepfake_roi" / "evaluation",
    SRC_ROOT / "deepfake_roi" / "utils",
]:
    p.mkdir(parents=True, exist_ok=True)

for init_file in [
    SRC_ROOT / "deepfake_roi" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "data" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "models" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "training" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "evaluation" / "__init__.py",
    SRC_ROOT / "deepfake_roi" / "utils" / "__init__.py",
]:
    init_file.write_text("", encoding="utf-8")

print("Workspace:", WORKSPACE)

In [ ]:
%%writefile /content/eyebrow_swinv2_experiment/src/deepfake_roi/utils/repro.py
from __future__ import annotations

import os
import random
from typing import Any, Dict

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def capture_rng_state() -> Dict[str, Any]:
    return {
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }


def restore_rng_state(state: Dict[str, Any]) -> None:
    random.setstate(state["python_rng_state"])
    np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"])

    if torch.cuda.is_available() and state.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(state["cuda_rng_state"])

In [ ]:
%%writefile /content/eyebrow_swinv2_experiment/src/deepfake_roi/utils/io.py
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
from typing import Any, Dict, Iterable

import torch


CHECKPOINT_REQUIRED_KEYS = {
    "epoch",
    "model_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "scaler_state_dict",
    "best_metric_score",
    "best_threshold",
    "best_epoch",
    "epochs_without_improvement",
    "history",
    "config",
    "rng_state",
    "loader_generator_state",
}


def atomic_save_checkpoint(state: Dict[str, Any], target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target.with_suffix(target.suffix + ".tmp")

    if temp_path.exists():
        temp_path.unlink()

    try:
        torch.save(state, temp_path)

        loaded = torch.load(
            temp_path,
            map_location="cpu",
            weights_only=False,
        )

        missing = CHECKPOINT_REQUIRED_KEYS.difference(loaded.keys())
        if missing:
            raise RuntimeError(
                f"Checkpoint integrity failed. Missing keys: {sorted(missing)}"
            )

        if int(loaded["epoch"]) != int(state["epoch"]):
            raise RuntimeError("Checkpoint epoch integrity mismatch.")

        os.replace(temp_path, target)

    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise


def load_checkpoint(path: Path, device: torch.device) -> Dict[str, Any]:
    if not path.is_file():
        raise FileNotFoundError(path)

    state = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    missing = CHECKPOINT_REQUIRED_KEYS.difference(state.keys())
    if missing:
        raise RuntimeError(
            f"Checkpoint incomplete. Missing keys: {sorted(missing)}"
        )

    return state


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def save_json_atomic(data: Dict[str, Any], target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    os.replace(tmp, target)


def hash_source_tree(paths: Iterable[Path]) -> Dict[str, str]:
    return {
        str(path): sha256_file(path)
        for path in sorted(paths)
        if path.is_file()
    }

In [ ]:
%%writefile /content/eyebrow_swinv2_experiment/src/deepfake_roi/data/dataset.py
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import Swin_V2_T_Weights


LABEL_TO_INDEX = {
    "real": 0.0,
    "fake": 1.0,
}


def build_transforms(
    image_size: int,
    augmentation: Dict[str, float],
):
    # Methodological choice:
    # For ImageNet-pretrained Swin V2, the normalization attached to the
    # pretrained weights is used. No validation/test statistics are learned.
    weights = Swin_V2_T_Weights.IMAGENET1K_V1
    weight_transform = weights.transforms()
    mean = weight_transform.mean
    std = weight_transform.std

    train_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(
                p=float(augmentation["horizontal_flip_probability"])
            ),
            transforms.RandomRotation(
                degrees=float(augmentation["rotation_degrees"])
            ),
            transforms.ColorJitter(
                brightness=float(augmentation["brightness"]),
                contrast=float(augmentation["contrast"]),
                saturation=float(augmentation["saturation"]),
                hue=float(augmentation["hue"]),
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )

    eval_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ]
    )

    normalization_info = {
        "source": "Swin_V2_T_Weights.IMAGENET1K_V1",
        "mean": list(mean),
        "std": list(std),
        "learned_from_project_data": False,
        "uses_validation_or_test_statistics": False,
    }

    return train_transform, eval_transform, normalization_info


class EyebrowROIDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform,
        sample_col: str,
        video_col: str,
        label_col: str,
        resolved_path_col: str = "_resolved_image_path",
    ) -> None:
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.sample_col = sample_col
        self.video_col = video_col
        self.label_col = label_col
        self.resolved_path_col = resolved_path_col

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        row = self.df.iloc[index]
        path = Path(str(row[self.resolved_path_col]))

        if not path.is_file():
            raise FileNotFoundError(
                f"ROI image disappeared after audit: {path}"
            )

        try:
            with Image.open(path) as img:
                image = img.convert("RGB")
        except Exception as exc:
            raise RuntimeError(f"Image decode failed: {path}") from exc

        image = self.transform(image)

        label_text = str(row[self.label_col])
        if label_text not in LABEL_TO_INDEX:
            raise ValueError(f"Unexpected label: {label_text}")

        return {
            "image": image,
            "label": torch.tensor(
                LABEL_TO_INDEX[label_text],
                dtype=torch.float32,
            ),
            "sample_id": str(row[self.sample_col]),
            "video_id": str(row[self.video_col]),
            "path": str(path),
        }

In [ ]:
%%writefile /content/eyebrow_swinv2_experiment/src/deepfake_roi/models/swin.py
from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import Swin_V2_T_Weights, swin_v2_t


class SwinV2TinyBinaryClassifier(nn.Module):
    def __init__(
        self,
        pretrained: bool = True,
        dropout: float = 0.20,
    ) -> None:
        super().__init__()

        weights = (
            Swin_V2_T_Weights.IMAGENET1K_V1
            if pretrained
            else None
        )

        self.backbone = swin_v2_t(weights=weights)

        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()

        self.classifier = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.classifier(features).squeeze(1)

In [ ]:
%%writefile /content/eyebrow_swinv2_experiment/src/deepfake_roi/training/engine.py
from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def safe_roc_auc(y_true: np.ndarray, probs: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, probs))


def safe_pr_auc(y_true: np.ndarray, probs: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(average_precision_score(y_true, probs))


def compute_metrics(
    y_true: np.ndarray,
    probs: np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    preds = (probs >= threshold).astype(np.int64)
    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "precision": float(
            precision_score(y_true, preds, zero_division=0)
        ),
        "recall": float(
            recall_score(y_true, preds, zero_division=0)
        ),
        "f1": float(
            f1_score(y_true, preds, zero_division=0)
        ),
        "roc_auc": safe_roc_auc(y_true, probs),
        "pr_auc": safe_pr_auc(y_true, probs),
    }


def select_best_f1_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    *,
    threshold_min: float,
    threshold_max: float,
    threshold_steps: int,
    default_threshold: float,
) -> Tuple[float, float]:
    if len(y_true) == 0:
        raise ValueError("Cannot select threshold from an empty validation set.")
    if threshold_steps < 2:
        raise ValueError("threshold_steps must be >= 2.")

    thresholds = np.linspace(
        float(threshold_min),
        float(threshold_max),
        int(threshold_steps),
    )
    best_threshold = float(default_threshold)
    best_f1 = -1.0

    for threshold in thresholds:
        preds = (probs >= threshold).astype(np.int64)
        score = f1_score(
            y_true,
            preds,
            zero_division=0,
        )
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_f1


def assert_finite_tensor(
    tensor: torch.Tensor,
    name: str,
) -> None:
    if not torch.isfinite(tensor).all():
        raise FloatingPointError(
            f"NaN/Inf detected in {name}."
        )


def assert_finite_gradients(model: nn.Module) -> None:
    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            continue
        if not torch.isfinite(parameter.grad).all():
            raise FloatingPointError(
                f"NaN/Inf gradient in parameter: {name}"
            )


def run_epoch(
    *,
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    amp_enabled: bool,
    scaler,
    gradient_clip_norm: float,
    amp_overflow_abort_after: int = 8,
    optimizer: Optional[torch.optim.Optimizer] = None,
) -> Dict[str, Any]:
    training = optimizer is not None
    model.train(training)

    running_loss = 0.0
    total_samples = 0
    amp_overflow_steps = 0
    consecutive_amp_overflows = 0

    labels_all: List[float] = []
    probs_all: List[float] = []
    sample_ids: List[str] = []
    video_ids: List[str] = []
    paths: List[str] = []

    grad_context = (
        torch.enable_grad()
        if training
        else torch.no_grad()
    )

    with grad_context:
        progress = tqdm(
            loader,
            leave=False,
            desc="Train" if training else "Evaluate",
        )

        for batch in progress:
            images = batch["image"].to(
                device,
                non_blocking=True,
            )
            labels = batch["label"].to(
                device,
                non_blocking=True,
            )

            assert_finite_tensor(images, "input images")
            assert_finite_tensor(labels, "labels")

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type,
                dtype=(
                    torch.float16
                    if device.type == "cuda"
                    else torch.bfloat16
                ),
                enabled=amp_enabled,
            ):
                logits = model(images)
                assert_finite_tensor(logits, "logits")
                loss = criterion(logits, labels)
                assert_finite_tensor(loss, "loss")

            if training:
                if amp_enabled:
                    # Dynamic loss scaling may transiently create Inf gradients.
                    # GradScaler is designed to skip that optimizer step and
                    # reduce its scale, so a single AMP overflow is not fatal.
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)

                    if gradient_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(),
                            max_norm=gradient_clip_norm,
                            error_if_nonfinite=False,
                        )

                    previous_scale = float(scaler.get_scale())
                    scaler.step(optimizer)
                    scaler.update()
                    current_scale = float(scaler.get_scale())

                    overflow = current_scale < previous_scale
                    if overflow:
                        amp_overflow_steps += 1
                        consecutive_amp_overflows += 1
                        optimizer.zero_grad(set_to_none=True)

                        if consecutive_amp_overflows >= amp_overflow_abort_after:
                            raise FloatingPointError(
                                "Persistent AMP gradient overflow: "
                                f"{consecutive_amp_overflows} consecutive steps. "
                                "Disable mixed precision or inspect inputs/model."
                            )
                    else:
                        consecutive_amp_overflows = 0

                else:
                    # FP32 baseline: non-finite gradients are always fatal.
                    loss.backward()
                    assert_finite_gradients(model)

                    if gradient_clip_norm > 0:
                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(),
                            max_norm=gradient_clip_norm,
                            error_if_nonfinite=True,
                        )

                    optimizer.step()

            probs = (
                torch.sigmoid(logits)
                .detach()
                .float()
                .cpu()
                .numpy()
            )
            labels_np = (
                labels.detach()
                .float()
                .cpu()
                .numpy()
            )

            batch_size = int(images.shape[0])
            running_loss += float(loss.item()) * batch_size
            total_samples += batch_size

            labels_all.extend(labels_np.tolist())
            probs_all.extend(probs.tolist())
            sample_ids.extend(list(batch["sample_id"]))
            video_ids.extend(list(batch["video_id"]))
            paths.extend(list(batch["path"]))

            postfix = {"loss": f"{loss.item():.4f}"}
            if amp_enabled:
                postfix["amp_overflows"] = amp_overflow_steps
            progress.set_postfix(**postfix)

    if total_samples == 0:
        raise RuntimeError("No samples processed.")

    return {
        "loss": running_loss / total_samples,
        "labels": np.asarray(labels_all, dtype=np.int64),
        "probabilities": np.asarray(probs_all, dtype=np.float64),
        "sample_ids": sample_ids,
        "video_ids": video_ids,
        "paths": paths,
        "amp_overflow_steps": int(amp_overflow_steps),
    }

In [ ]:
%%writefile /content/eyebrow_swinv2_experiment/src/deepfake_roi/evaluation/plots.py
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)


def save_figure(
    fig,
    output_path: Path,
    min_short_edge: int = 600,
) -> None:
    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)

    with Image.open(output_path) as image:
        if min(image.size) < min_short_edge:
            raise RuntimeError(
                f"Figure resolution failed: "
                f"{output_path} -> {image.size}"
            )


def generate_all_figures(
    *,
    history_df,
    labels,
    probabilities,
    threshold: float,
    figures_dir: Path,
    min_short_edge: int,
) -> None:
    figures_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Training Loss",
        linewidth=2,
    )
    ax.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label="Validation Loss",
        linewidth=2,
        linestyle="--",
    )
    ax.set_title(
        "Training and Validation Loss Curve",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "training_validation_loss.png",
        min_short_edge,
    )

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(
        history_df["epoch"],
        history_df["train_accuracy"],
        label="Training Accuracy",
        linewidth=2,
    )
    ax.plot(
        history_df["epoch"],
        history_df["val_accuracy"],
        label="Validation Accuracy",
        linewidth=2,
        linestyle="--",
    )
    ax.set_title(
        "Training and Validation Accuracy",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("Accuracy", fontsize=11)
    ax.set_ylim(0, 1.01)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "training_validation_accuracy.png",
        min_short_edge,
    )

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(
        history_df["epoch"],
        history_df["val_roc_auc"],
        label="Validation ROC-AUC",
        linewidth=2,
    )
    ax.set_title(
        "Validation ROC-AUC by Epoch",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("ROC-AUC", fontsize=11)
    ax.set_ylim(0, 1.01)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "validation_roc_auc.png",
        min_short_edge,
    )

    preds = (probabilities >= threshold).astype(np.int64)
    cm = confusion_matrix(
        labels,
        preds,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
    image = ax.imshow(cm)
    ax.set_title(
        "Test Confusion Matrix",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Predicted Class", fontsize=11)
    ax.set_ylabel("True Class", fontsize=11)
    ax.set_xticks([0, 1], labels=["Real", "Fake"])
    ax.set_yticks([0, 1], labels=["Real", "Fake"])

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                fontsize=14,
            )

    fig.colorbar(image, ax=ax)
    save_figure(
        fig,
        figures_dir / "test_confusion_matrix.png",
        min_short_edge,
    )

    if len(np.unique(labels)) == 2:
        fpr, tpr, _ = roc_curve(labels, probabilities)
        auc = roc_auc_score(labels, probabilities)

        fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"Swin V2 Tiny (AUC = {auc:.4f})",
        )
        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1.5,
            label="Random Classifier",
        )
        ax.set_title(
            "Test ROC Curve",
            fontsize=14,
            fontweight="bold",
        )
        ax.set_xlabel(
            "False Positive Rate",
            fontsize=11,
        )
        ax.set_ylabel(
            "True Positive Rate",
            fontsize=11,
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.01)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.25)
        save_figure(
            fig,
            figures_dir / "test_roc_curve.png",
            min_short_edge,
        )

    precision, recall, _ = precision_recall_curve(
        labels,
        probabilities,
    )
    ap = average_precision_score(
        labels,
        probabilities,
    )

    fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
    ax.plot(
        recall,
        precision,
        linewidth=2,
        label=f"Swin V2 Tiny (AP = {ap:.4f})",
    )
    ax.set_title(
        "Test Precision-Recall Curve",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Recall", fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)
    save_figure(
        fig,
        figures_dir / "test_precision_recall_curve.png",
        min_short_edge,
    )

In [ ]:
# ============================================================
# 3. IMPORT MODULAR PACKAGE
# ============================================================
import sys

sys.path.insert(0, str(SRC_ROOT))

from deepfake_roi.data.dataset import EyebrowROIDataset, build_transforms
from deepfake_roi.evaluation.plots import generate_all_figures
from deepfake_roi.models.swin import SwinV2TinyBinaryClassifier
from deepfake_roi.training.engine import (
    assert_finite_gradients,
    assert_finite_tensor,
    compute_metrics,
    run_epoch,
    select_best_f1_threshold,
)
from deepfake_roi.utils.io import (
    CHECKPOINT_REQUIRED_KEYS,
    atomic_save_checkpoint,
    hash_source_tree,
    load_checkpoint,
    save_json_atomic,
    sha256_file,
)
from deepfake_roi.utils.repro import (
    capture_rng_state,
    restore_rng_state,
    seed_everything,
    seed_worker,
)

print("Modular package import: PASSED")

In [ ]:
# ============================================================
# 4. SINGLE SOURCE OF TRUTH — YAML CONFIG
# ============================================================
from pathlib import Path
import yaml

CONFIG_YAML_PATH = Path("/content/eyebrow_swinv2_tiny_experiment.yaml")

CONFIG_YAML = r"""
experiment:
  region: eyebrow
  model_name: swinv2_tiny
  seed: 42
  notebook_version: v4_eyebrow_pdf_standard
  mode: new
  resume_run_id: null

paths:
  project_root_candidates:
    - "/content/drive/MyDrive/AISC DeepFake Çalışmaları"
    - "/content/drive/.shortcut-targets-by-id/1qDJf3MYGlyKCNCfp7hS82eXp9H8F1Br1/AISC DeepFake Çalışmaları"
  eyebrow_roi_relative: "Deneyler/Nazlıcan/Deney 1/Kaş"
  selection_metadata_relative: "Deneyler/Deney 1 Frame/secim_metadata.csv"
  results_relative: "Deneyler/Nazlıcan/Deney 1/Sonuçlar"

data:
  roi_metadata_filename: metadata.csv
  sample_id_column: sample_id
  label_column: label
  split_column: split
  status_column: status
  relative_path_column: output_relative_path
  absolute_path_column: output_path
  input_path_column: input_path
  output_sha256_column: output_sha256
  accepted_status: [SUCCESS, OK]
  allowed_labels: [real, fake]
  allowed_splits: [train, val, test]
  image_size: 224
  batch_size: 32
  num_workers: 2
  pin_memory: true
  expected_split_ratios:
    train: 0.80
    val: 0.10
    test: 0.10
  split_ratio_tolerance: 0.06

selection_metadata:
  class_column: sinif
  split_column: split
  original_path_column: orijinal_yol
  selected_path_column: yeni_yol
  filename_column: dosya_adi

normalization:
  source: pretrained_weights
  weights: Swin_V2_T_Weights.IMAGENET1K_V1
  rationale: >-
    ImageNet-pretrained Swin V2'nin kendi normalizasyonu kullanılır.
    Proje train/val/test verisinden mean/std öğrenilmez; validation veya
    test istatistiği normalizasyona karışmaz.

model:
  pretrained: true
  dropout: 0.20

training:
  epochs: 20
  backbone_lr: 1.0e-5
  head_lr: 1.0e-4
  weight_decay: 1.0e-2
  gradient_clip_norm: 1.0
  mixed_precision: false
  amp_overflow_abort_after: 8
  early_stopping_patience: 6
  monitor_metric: roc_auc
  fallback_metric: f1
  keep_last_n_epoch_checkpoints: 3
  scheduler_eta_min_factor: 0.10
  min_improvement: 1.0e-6
  default_threshold: 0.50
  threshold_search_min: 0.05
  threshold_search_max: 0.95
  threshold_search_steps: 181

augmentation:
  horizontal_flip_probability: 0.50
  rotation_degrees: 5
  brightness: 0.10
  contrast: 0.10
  saturation: 0.05
  hue: 0.02

quality:
  smoke_test_batches: 2
  smoke_test_lr: 1.0e-6
  min_figure_short_edge_px: 600
  checkpoint_continuity_atol: 1.0e-7
  checkpoint_continuity_rtol: 1.0e-6
"""

# Training code reads its settings from YAML, not from scattered hyperparameters.
CONFIG_YAML_PATH.write_text(CONFIG_YAML, encoding="utf-8")
with CONFIG_YAML_PATH.open("r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

print("SSOT config:", CONFIG_YAML_PATH)
print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))


In [ ]:
# ============================================================
# 5. RESOLVE EXACT DRIVE PATHS + NEW/RESUME RUN
# ============================================================
import json
import logging
import os
import random
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import PIL
import sklearn
import torch
import torchvision
import yaml


def first_existing_path(candidates):
    existing = [Path(p) for p in candidates if Path(p).exists()]
    if not existing:
        raise FileNotFoundError(
            "AISC DeepFake Çalışmaları klasörü bulunamadı.\n"
            "Kontrol edilen yollar:\n- " + "\n- ".join(candidates)
        )
    return existing[0]


PROJECT_ROOT = first_existing_path(CONFIG["paths"]["project_root_candidates"])
EYEBROW_ROI_ROOT = PROJECT_ROOT / CONFIG["paths"]["eyebrow_roi_relative"]
METADATA_PATH = EYEBROW_ROI_ROOT / CONFIG["data"]["roi_metadata_filename"]
SELECTION_METADATA_PATH = PROJECT_ROOT / CONFIG["paths"]["selection_metadata_relative"]
RESULTS_ROOT = PROJECT_ROOT / CONFIG["paths"]["results_relative"]

required_inputs = [EYEBROW_ROI_ROOT, METADATA_PATH, SELECTION_METADATA_PATH]
missing_inputs = [str(p) for p in required_inputs if not p.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Required eyebrow experiment input is missing:\n" + "\n".join(missing_inputs)
    )

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
mode = str(CONFIG["experiment"]["mode"]).lower().strip()
if mode not in {"new", "resume"}:
    raise ValueError("experiment.mode must be 'new' or 'resume'.")

SEED = int(CONFIG["experiment"]["seed"])

if mode == "new":
    run_id = (
        f"{datetime.now():%Y%m%d_%H%M}_"
        f"{CONFIG['experiment']['region']}_"
        f"{CONFIG['experiment']['model_name']}_seed{SEED}"
    )
    RUN_DIR = RESULTS_ROOT / run_id
    if RUN_DIR.exists():
        raise FileExistsError(
            f"Run directory already exists and will not be overwritten: {RUN_DIR}. "
            "Wait for a new minute or use resume mode with this run_id."
        )
    RUN_DIR.mkdir(parents=True, exist_ok=False)
else:
    resume_run_id = CONFIG["experiment"]["resume_run_id"]
    if not resume_run_id:
        raise ValueError("Resume mode selected but experiment.resume_run_id is empty.")
    run_id = str(resume_run_id)
    RUN_DIR = RESULTS_ROOT / run_id
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f"Resume run directory not found: {RUN_DIR}")

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
METRICS_DIR = RUN_DIR / "metrics"
PREDICTIONS_DIR = RUN_DIR / "predictions"
FIGURES_DIR = RUN_DIR / "figures"
ARTIFACTS_DIR = RUN_DIR / "artifacts"
SOURCE_DIR = ARTIFACTS_DIR / "source_snapshot"

for directory in [
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    FIGURES_DIR,
    ARTIFACTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Structured log file.
logger = logging.getLogger(f"eyebrow_swin_{run_id}")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "pipeline.log", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

resolved_name = (
    "config_resolved.yaml"
    if mode == "new"
    else f"config_resume_{datetime.now():%Y%m%d_%H%M%S}.yaml"
)
with (RUN_DIR / resolved_name).open("w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False, allow_unicode=True)

logger.info("Mode: %s", mode)
logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Eyebrow ROI root: %s", EYEBROW_ROI_ROOT)
logger.info("ROI metadata: %s", METADATA_PATH)
logger.info("Selection metadata: %s", SELECTION_METADATA_PATH)
logger.info("Run directory: %s", RUN_DIR)


In [ ]:
# ============================================================
# 6. ENVIRONMENT LOCK + SOURCE SNAPSHOT/HASH
# ============================================================
import importlib.metadata as md

packages_to_lock = [
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "Pillow",
    "PyYAML",
    "tqdm",
]

lock_lines = []
for package in packages_to_lock:
    try:
        lock_lines.append(f"{package}=={md.version(package)}")
    except md.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {package}")

requirements_lock = RUN_DIR / "requirements_lock.txt"
requirements_lock.write_text("\n".join(lock_lines) + "\n", encoding="utf-8")

environment_manifest = {
    "run_id": run_id,
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_version": torch.version.cuda,
    "seed": SEED,
    "requirements_lock": "requirements_lock.txt",
}
save_json_atomic(environment_manifest, RUN_DIR / "environment.json")

if mode == "new":
    shutil.copytree(
        SRC_ROOT / "deepfake_roi",
        SOURCE_DIR / "deepfake_roi",
        dirs_exist_ok=False,
    )

source_files = list((SOURCE_DIR / "deepfake_roi").rglob("*.py")) if SOURCE_DIR.exists() else []
source_hashes = hash_source_tree(source_files)
save_json_atomic(
    {"run_id": run_id, "files": source_hashes},
    ARTIFACTS_DIR / "source_hash_manifest.json",
)

logger.info("Environment lock written: %s", requirements_lock)
logger.info("Source snapshot files: %d", len(source_files))
print(requirements_lock.read_text())


In [ ]:
# ============================================================
# 7. REPRODUCIBILITY
# ============================================================
seed_everything(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 8. EYEBROW METADATA + ACCOUNTING + SOURCE-VIDEO LEAKAGE GATES
# ============================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd

D = CONFIG["data"]
S = CONFIG["selection_metadata"]

roi_metadata_raw = pd.read_csv(METADATA_PATH)
selection_metadata_raw = pd.read_csv(SELECTION_METADATA_PATH)

sample_col = D["sample_id_column"]
label_col = D["label_column"]
split_col = D["split_column"]
status_col = D["status_column"]
video_col = "video_id"
image_col = D["relative_path_column"]

# ---------- Schema gate: source files ----------
roi_required = {
    sample_col,
    label_col,
    split_col,
    status_col,
    D["relative_path_column"],
    D["absolute_path_column"],
    D["input_path_column"],
    "frame_index",
    "face_index",
    "skip_reason",
    "run_id",
}
selection_required = {
    S["class_column"],
    S["split_column"],
    S["original_path_column"],
    S["filename_column"],
}

missing_roi = sorted(roi_required - set(roi_metadata_raw.columns))
missing_selection = sorted(selection_required - set(selection_metadata_raw.columns))
if missing_roi or missing_selection:
    raise ValueError(
        f"Schema gate failed. ROI missing={missing_roi}; "
        f"selection metadata missing={missing_selection}"
    )

# PDF canonical fields vs actual upstream schema. Semantic aliases are documented,
# never silently invented.
canonical_aliases = {
    "source_video": "recovered from secim_metadata.orijinal_yol parent folder",
    "sha256": D["output_sha256_column"],
    "output_path": "resolved from output_relative_path under Kaş root",
    "roi_state": "not applicable to this static eyebrow ROI baseline",
}

schema_audit = {
    "roi_metadata_columns": list(roi_metadata_raw.columns),
    "selection_metadata_columns": list(selection_metadata_raw.columns),
    "canonical_aliases_or_provenance": canonical_aliases,
    "training_schema_gate_passed": True,
    "note": (
        "source_video is not guessed from the eyebrow metadata because it is blank in "
        "the upstream ROI table; it is recovered deterministically from selection metadata."
    ),
}
save_json_atomic(schema_audit, ARTIFACTS_DIR / "metadata_schema_audit.json")

# ---------- Accounting equality on upstream eyebrow ROI pipeline ----------
roi_metadata_raw = roi_metadata_raw.copy()
roi_metadata_raw[status_col] = (
    roi_metadata_raw[status_col].astype(str).str.strip().str.upper()
)
status_counts = roi_metadata_raw[status_col].value_counts(dropna=False).to_dict()
accepted_status = {str(x).upper() for x in D["accepted_status"]}
known_status = accepted_status | {"SKIPPED", "ERROR"}
unknown_statuses = sorted(set(status_counts) - known_status)
if unknown_statuses:
    raise ValueError(f"Unknown ROI status values: {unknown_statuses}")

total_inputs = int(len(roi_metadata_raw))
success_count = int(roi_metadata_raw[status_col].isin(accepted_status).sum())
skipped_count = int((roi_metadata_raw[status_col] == "SKIPPED").sum())
error_count = int((roi_metadata_raw[status_col] == "ERROR").sum())
assert total_inputs == success_count + skipped_count + error_count, (
    "Accounting equality failed: total != success + skipped + error"
)
assert roi_metadata_raw[sample_col].notna().all(), "Missing sample_id found."
assert roi_metadata_raw[sample_col].is_unique, "Duplicate sample_id found."

accounting = {
    "total_inputs": total_inputs,
    "success_count": success_count,
    "skipped_count": skipped_count,
    "error_count": error_count,
    "status_counts": {str(k): int(v) for k, v in status_counts.items()},
    "accounting_equality_passed": True,
}
save_json_atomic(accounting, ARTIFACTS_DIR / "data_accounting.json")

# ---------- Keep only usable ROI rows ----------
metadata = roi_metadata_raw[
    roi_metadata_raw[status_col].isin(accepted_status)
].copy()
if metadata.empty:
    raise RuntimeError("No SUCCESS/OK eyebrow ROI rows available for training.")

metadata[label_col] = metadata[label_col].astype(str).str.strip().str.lower()
metadata[split_col] = (
    metadata[split_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"training": "train", "validation": "val", "valid": "val", "testing": "test"})
)

allowed_labels = set(D["allowed_labels"])
allowed_splits = set(D["allowed_splits"])
if set(metadata[label_col].unique()) - allowed_labels:
    raise ValueError(f"Unexpected labels: {sorted(set(metadata[label_col]) - allowed_labels)}")
if set(metadata[split_col].unique()) - allowed_splits:
    raise ValueError(f"Unexpected splits: {sorted(set(metadata[split_col]) - allowed_splits)}")
if allowed_splits - set(metadata[split_col].unique()):
    raise ValueError(f"Missing splits: {sorted(allowed_splits - set(metadata[split_col].unique()))}")

for split_name in sorted(allowed_splits):
    split_labels = set(metadata.loc[metadata[split_col] == split_name, label_col].unique())
    if split_labels != allowed_labels:
        raise ValueError(
            f"Split {split_name!r} must contain both real and fake. Found: {sorted(split_labels)}"
        )

# ---------- Resolve every ROI image under the exact Kaş folder ----------
def resolve_eyebrow_roi_path(relative_value, absolute_value):
    if pd.notna(relative_value) and str(relative_value).strip():
        candidate = EYEBROW_ROI_ROOT / str(relative_value).strip()
        return candidate
    if pd.notna(absolute_value) and str(absolute_value).strip():
        absolute = Path(str(absolute_value).strip())
        if absolute.is_file():
            return absolute
    raise ValueError("ROI row has neither a usable relative nor absolute output path.")

metadata["_resolved_image_path"] = [
    str(resolve_eyebrow_roi_path(rel, abs_path))
    for rel, abs_path in zip(
        metadata[D["relative_path_column"]],
        metadata[D["absolute_path_column"]],
    )
]
exists_mask = metadata["_resolved_image_path"].map(lambda p: Path(p).is_file())
if not exists_mask.all():
    missing_file = ARTIFACTS_DIR / "missing_success_roi_files.csv"
    metadata.loc[~exists_mask].to_csv(missing_file, index=False)
    raise FileNotFoundError(
        f"{int((~exists_mask).sum())} SUCCESS/OK eyebrow ROI files are missing. Audit: {missing_file}"
    )

# ---------- Recover true source-video identity from Deney 1 Frame metadata ----------
metadata["frame_name"] = metadata[D["input_path_column"]].map(
    lambda value: Path(str(value)).name
)
selection = selection_metadata_raw.copy()
selection["frame_name"] = selection[S["filename_column"]].astype(str).str.strip()

if selection["frame_name"].duplicated().any():
    duplicate_selection = selection[selection["frame_name"].duplicated(keep=False)]
    duplicate_selection.to_csv(ARTIFACTS_DIR / "selection_duplicate_frame_names.csv", index=False)
    raise ValueError("Selection metadata has duplicate dosya_adi/frame_name values.")

selection["selection_label"] = (
    selection[S["class_column"]].astype(str).str.strip().str.lower()
)
selection["selection_split"] = (
    selection[S["split_column"]]
    .astype(str).str.strip().str.lower()
    .replace({"training": "train", "validation": "val", "valid": "val", "testing": "test"})
)
selection["source_video_recovered"] = selection[S["original_path_column"]].map(
    lambda value: Path(str(value)).parent.name if pd.notna(value) and str(value).strip() else ""
)

metadata = metadata.merge(
    selection[[
        "frame_name",
        "selection_label",
        "selection_split",
        "source_video_recovered",
        S["original_path_column"],
    ]],
    on="frame_name",
    how="left",
    validate="many_to_one",
    indicator=True,
)

unmatched = metadata[metadata["_merge"] != "both"].copy()
if not unmatched.empty:
    unmatched_file = ARTIFACTS_DIR / "metadata_unmatched_rows.csv"
    unmatched.to_csv(unmatched_file, index=False)
    raise ValueError(
        f"{len(unmatched)} eyebrow ROI rows cannot be mapped to selection metadata: {unmatched_file}"
    )
metadata.drop(columns=["_merge"], inplace=True)

label_mismatch = metadata[label_col] != metadata["selection_label"]
split_mismatch = metadata[split_col] != metadata["selection_split"]
if label_mismatch.any() or split_mismatch.any():
    mismatch_file = ARTIFACTS_DIR / "metadata_label_split_mismatch.csv"
    metadata.loc[label_mismatch | split_mismatch].to_csv(mismatch_file, index=False)
    raise ValueError(f"ROI vs selection metadata label/split mismatch: {mismatch_file}")

if (metadata["source_video_recovered"].astype(str).str.strip() == "").any():
    bad_file = ARTIFACTS_DIR / "missing_recovered_source_video.csv"
    metadata.loc[
        metadata["source_video_recovered"].astype(str).str.strip() == ""
    ].to_csv(bad_file, index=False)
    raise ValueError(f"Source video recovery failed: {bad_file}")

metadata["source_video"] = metadata["source_video_recovered"].astype(str).str.strip()
metadata[video_col] = metadata[label_col] + "__" + metadata["source_video"]

# Canonical sha256 alias uses the existing output hash; values are not fabricated.
sha_col = D["output_sha256_column"]
if sha_col in metadata.columns:
    metadata["sha256"] = metadata[sha_col]
else:
    metadata["sha256"] = ""

# ---------- Source-video leakage gate ----------
video_label_counts = metadata.groupby(video_col)[label_col].nunique()
if (video_label_counts > 1).any():
    bad_ids = video_label_counts[video_label_counts > 1].index
    metadata[metadata[video_col].isin(bad_ids)].to_csv(
        ARTIFACTS_DIR / "video_label_conflicts.csv", index=False
    )
    raise ValueError("A source video maps to multiple labels.")

video_split_counts = metadata.groupby(video_col)[split_col].nunique()
if (video_split_counts > 1).any():
    bad_ids = video_split_counts[video_split_counts > 1].index
    metadata[metadata[video_col].isin(bad_ids)].to_csv(
        ARTIFACTS_DIR / "video_split_conflicts.csv", index=False
    )
    raise ValueError("A source video appears in multiple splits.")

train_videos = set(metadata.loc[metadata[split_col] == "train", video_col])
val_videos = set(metadata.loc[metadata[split_col] == "val", video_col])
test_videos = set(metadata.loc[metadata[split_col] == "test", video_col])

assert train_videos.isdisjoint(val_videos), "Train/Val source-video leakage detected."
assert train_videos.isdisjoint(test_videos), "Train/Test source-video leakage detected."
assert val_videos.isdisjoint(test_videos), "Val/Test source-video leakage detected."

# Check expected 80/10/10 at source-video level independently per class.
expected_ratios = D["expected_split_ratios"]
tolerance = float(D["split_ratio_tolerance"])
split_ratio_audit = {}
for class_name in sorted(allowed_labels):
    class_video_table = (
        metadata.loc[metadata[label_col] == class_name, [video_col, split_col]]
        .drop_duplicates()
    )
    total_class_videos = len(class_video_table)
    if total_class_videos == 0:
        raise RuntimeError(f"No source videos for class {class_name}")
    observed = {
        split_name: float((class_video_table[split_col] == split_name).sum() / total_class_videos)
        for split_name in sorted(allowed_splits)
    }
    split_ratio_audit[class_name] = {
        "source_video_count": total_class_videos,
        "observed": observed,
        "expected": expected_ratios,
    }
    for split_name, expected in expected_ratios.items():
        if abs(observed[split_name] - float(expected)) > tolerance:
            raise ValueError(
                f"{class_name} source-video split ratio for {split_name} is {observed[split_name]:.3f}; "
                f"expected {float(expected):.3f} ± {tolerance:.3f}."
            )

video_identity_reliable = True
split_report = {
    "passed": True,
    "train_video_count": len(train_videos),
    "val_video_count": len(val_videos),
    "test_video_count": len(test_videos),
    "train_val_overlap": 0,
    "train_test_overlap": 0,
    "val_test_overlap": 0,
    "class_source_video_split_ratios": split_ratio_audit,
}
save_json_atomic(split_report, ARTIFACTS_DIR / "split_leakage_report.json")

# ---------- Persist exactly what will be consumed ----------
metadata["output_path"] = metadata["_resolved_image_path"]
metadata["roi_state"] = "not_applicable_static_eyebrow_roi"
metadata.to_csv(ARTIFACTS_DIR / "metadata_used.csv", index=False)

summary_rows = []
for split_name in ["train", "val", "test"]:
    for class_name in ["real", "fake"]:
        subset = metadata[(metadata[split_col] == split_name) & (metadata[label_col] == class_name)]
        summary_rows.append({
            "split": split_name,
            "label": class_name,
            "frame_count": int(len(subset)),
            "source_video_count": int(subset[video_col].nunique()),
        })
pd.DataFrame(summary_rows).to_csv(ARTIFACTS_DIR / "dataset_summary.csv", index=False)

ssot_missing = []
logger.info("Metadata/accounting/source-video leakage gates PASSED")
logger.info("Training rows: %d", len(metadata))
print(pd.DataFrame(summary_rows))
print(json.dumps(split_report, indent=2))


In [ ]:
# ============================================================
# 9. NORMALIZATION AUDIT + DATASET/DATALOADERS
# ============================================================
train_transform, eval_transform, normalization_info = build_transforms(
    image_size=int(D["image_size"]),
    augmentation=CONFIG["augmentation"],
)

normalization_audit = {
    **normalization_info,
    "config_source": CONFIG["normalization"]["source"],
    "rationale": CONFIG["normalization"]["rationale"],
}

save_json_atomic(
    normalization_audit,
    ARTIFACTS_DIR / "normalization_audit.json",
)

if normalization_info["uses_validation_or_test_statistics"]:
    raise RuntimeError(
        "Normalization leakage detected."
    )

train_df = metadata[
    metadata[split_col] == "train"
].copy()
val_df = metadata[
    metadata[split_col] == "val"
].copy()
test_df = metadata[
    metadata[split_col] == "test"
].copy()

for name, frame in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    if frame.empty:
        raise RuntimeError(f"{name} split is empty.")

train_dataset = EyebrowROIDataset(
    train_df,
    train_transform,
    sample_col,
    video_col,
    label_col,
)
val_dataset = EyebrowROIDataset(
    val_df,
    eval_transform,
    sample_col,
    video_col,
    label_col,
)
test_dataset = EyebrowROIDataset(
    test_df,
    eval_transform,
    sample_col,
    video_col,
    label_col,
)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

loader_kwargs = {
    "batch_size": int(D["batch_size"]),
    "num_workers": int(D["num_workers"]),
    "pin_memory": (
        bool(D["pin_memory"])
        and DEVICE.type == "cuda"
    ),
    "worker_init_fn": seed_worker,
    "generator": loader_generator,
    "persistent_workers": (
        int(D["num_workers"]) > 0
    ),
}

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=False,
    **loader_kwargs,
)

# Validation/test için farklı generator gerekmez; shuffle=False.
eval_loader_kwargs = dict(loader_kwargs)
eval_loader_kwargs["generator"] = None

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **eval_loader_kwargs,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **eval_loader_kwargs,
)

print("Train / Val / Test frames:",
      len(train_df), len(val_df), len(test_df))
print("Normalization:", normalization_audit)

In [ ]:
# ============================================================
# 10. MODEL + TRAIN-ONLY LOSS WEIGHT + OPTIMIZER
# ============================================================
model = SwinV2TinyBinaryClassifier(
    pretrained=bool(CONFIG["model"]["pretrained"]),
    dropout=float(CONFIG["model"]["dropout"]),
).to(DEVICE)

train_counts = train_df[label_col].value_counts()

num_real = int(train_counts.get("real", 0))
num_fake = int(train_counts.get("fake", 0))

if num_real == 0 or num_fake == 0:
    raise RuntimeError(
        "Training split must contain both classes."
    )

# Train-only statistic.
pos_weight_value = num_real / num_fake
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE,
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

T = CONFIG["training"]

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.backbone.parameters(),
            "lr": float(T["backbone_lr"]),
        },
        {
            "params": model.classifier.parameters(),
            "lr": float(T["head_lr"]),
        },
    ],
    weight_decay=float(T["weight_decay"]),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=int(T["epochs"]),
    eta_min=float(T["backbone_lr"]) * float(T["scheduler_eta_min_factor"]),
)

AMP_ENABLED = (
    bool(T["mixed_precision"])
    and DEVICE.type == "cuda"
)

scaler_device = "cuda" if DEVICE.type == "cuda" else "cpu"
scaler = torch.amp.GradScaler(
    scaler_device,
    enabled=AMP_ENABLED,
)

print("Train real:", num_real)
print("Train fake:", num_fake)
print("pos_weight:", pos_weight_value)
print("AMP:", AMP_ENABLED)

In [ ]:
# ============================================================
# 11. INPUT SANITY + FP32 SMOKE TEST
# ============================================================
import copy
import torch

# Numerical input sanity gate before any optimizer update.
diagnostic_batch = next(iter(train_loader))
diagnostic_images = diagnostic_batch["image"]
diagnostic_labels = diagnostic_batch["label"]

input_report = {
    "shape": list(diagnostic_images.shape),
    "dtype": str(diagnostic_images.dtype),
    "min": float(diagnostic_images.min().item()),
    "max": float(diagnostic_images.max().item()),
    "mean": float(diagnostic_images.mean().item()),
    "std": float(diagnostic_images.std().item()),
    "images_finite": bool(torch.isfinite(diagnostic_images).all().item()),
    "labels_finite": bool(torch.isfinite(diagnostic_labels).all().item()),
}

save_json_atomic(
    input_report,
    ARTIFACTS_DIR / "input_numerical_sanity.json",
)

if not input_report["images_finite"]:
    raise FloatingPointError("NaN/Inf detected in input images.")
if not input_report["labels_finite"]:
    raise FloatingPointError("NaN/Inf detected in labels.")

print("Input numerical sanity: PASSED")
print(input_report)


def run_smoke_test(
    base_model,
    loader,
    max_batches: int,
):
    # The quality gate intentionally runs in FP32 regardless of the later
    # optional AMP setting. This distinguishes model/data instability from
    # mixed-precision overflow.
    smoke_model = copy.deepcopy(base_model).to(DEVICE)
    smoke_optimizer = torch.optim.AdamW(
        smoke_model.parameters(),
        lr=float(CONFIG["quality"]["smoke_test_lr"]),
    )
    smoke_model.train()

    completed = 0

    for batch in loader:
        images = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        assert_finite_tensor(images, "smoke input")
        assert_finite_tensor(labels, "smoke labels")

        smoke_optimizer.zero_grad(set_to_none=True)

        # No autocast here: strict FP32 diagnostic.
        logits = smoke_model(images)
        assert_finite_tensor(logits, "smoke logits")

        loss = criterion(logits, labels)
        assert_finite_tensor(loss, "smoke test loss")

        loss.backward()
        assert_finite_gradients(smoke_model)

        torch.nn.utils.clip_grad_norm_(
            smoke_model.parameters(),
            max_norm=float(T["gradient_clip_norm"]),
            error_if_nonfinite=True,
        )
        smoke_optimizer.step()

        completed += 1
        print(
            f"FP32 smoke batch {completed}: "
            f"loss={loss.item():.8f}"
        )

        if completed >= max_batches:
            break

    del smoke_model
    del smoke_optimizer

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    if completed < max_batches:
        raise RuntimeError(
            "FP32 smoke test could not complete."
        )

    print("FP32 smoke test: PASSED")


run_smoke_test(
    model,
    train_loader,
    int(CONFIG["quality"]["smoke_test_batches"]),
)

In [ ]:
# ============================================================
# 12. CHECKPOINT BUILD + CONTINUITY QUALITY GATE
# ============================================================

import torch

BEST_CHECKPOINT = CHECKPOINT_DIR / "best.ckpt"
LAST_CHECKPOINT = CHECKPOINT_DIR / "last.ckpt"

def build_checkpoint_state(
    *,
    epoch,
    best_metric_score,
    best_threshold,
    history,
    best_epoch,
    epochs_without_improvement,
):
    return {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_metric_score": float(best_metric_score),
        "best_threshold": float(best_threshold),
        "best_epoch": int(best_epoch),
        "epochs_without_improvement": int(epochs_without_improvement),
        "history": history,
        "config": CONFIG,
        "run_id": run_id,
        "rng_state": capture_rng_state(),
        "loader_generator_state": loader_generator.get_state(),
    }


def checkpoint_continuity_test():
    # Deterministic validation batch: no augmentation, model.eval() disables dropout.
    batch = next(iter(val_loader))
    images = batch["image"].to(DEVICE)
    labels = batch["label"].to(DEVICE)

    original_training_mode = model.training
    model.eval()

    with torch.no_grad():
        logits_before = model(images)
        loss_before = criterion(
            logits_before,
            labels,
        )

    temporary = (
        CHECKPOINT_DIR /
        "_continuity_test.ckpt"
    )

    state = build_checkpoint_state(
        epoch=0,
        best_metric_score=-1.0,
        best_threshold=0.5,
        history=[],
        best_epoch=0,
        epochs_without_improvement=0,
    )

    atomic_save_checkpoint(
        state,
        temporary,
    )

    fresh_model = SwinV2TinyBinaryClassifier(
        pretrained=False,
        dropout=float(CONFIG["model"]["dropout"]),
    ).to(DEVICE)

    fresh_optimizer = torch.optim.AdamW(
        [
            {
                "params": fresh_model.backbone.parameters(),
                "lr": float(T["backbone_lr"]),
            },
            {
                "params": fresh_model.classifier.parameters(),
                "lr": float(T["head_lr"]),
            },
        ],
        weight_decay=float(T["weight_decay"]),
    )

    fresh_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        fresh_optimizer,
        T_max=int(T["epochs"]),
        eta_min=float(T["backbone_lr"]) * float(T["scheduler_eta_min_factor"]),
    )

    fresh_scaler = torch.amp.GradScaler(
        "cuda" if DEVICE.type == "cuda" else "cpu",
        enabled=AMP_ENABLED,
    )

    loaded = load_checkpoint(
        temporary,
        DEVICE,
    )

    fresh_model.load_state_dict(
        loaded["model_state_dict"]
    )
    fresh_optimizer.load_state_dict(
        loaded["optimizer_state_dict"]
    )
    fresh_scheduler.load_state_dict(
        loaded["scheduler_state_dict"]
    )
    fresh_scaler.load_state_dict(
        loaded["scaler_state_dict"]
    )

    fresh_model.eval()

    with torch.no_grad():
        logits_after = fresh_model(images)
        loss_after = criterion(
            logits_after,
            labels,
        )

    atol = float(
        CONFIG["quality"]["checkpoint_continuity_atol"]
    )
    rtol = float(
        CONFIG["quality"]["checkpoint_continuity_rtol"]
    )

    if not torch.allclose(
        logits_before,
        logits_after,
        atol=atol,
        rtol=rtol,
    ):
        raise RuntimeError(
            "Checkpoint continuity FAILED: logits mismatch."
        )

    if not torch.allclose(
        loss_before,
        loss_after,
        atol=atol,
        rtol=rtol,
    ):
        raise RuntimeError(
            "Checkpoint continuity FAILED: loss mismatch."
        )

    report = {
        "passed": True,
        "loss_before": float(loss_before.item()),
        "loss_after": float(loss_after.item()),
        "absolute_loss_difference": float(
            abs(loss_before.item() - loss_after.item())
        ),
        "atol": atol,
        "rtol": rtol,
    }

    save_json_atomic(
        report,
        ARTIFACTS_DIR / "checkpoint_continuity_test.json",
    )

    temporary.unlink()

    if original_training_mode:
        model.train()

    del fresh_model
    del fresh_optimizer
    del fresh_scheduler
    del fresh_scaler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print("Checkpoint continuity test: PASSED")
    print(report)


checkpoint_continuity_test()


In [ ]:
# ============================================================
# 13. TRUE RESUME LOGIC
# ============================================================

import numpy as np

history = []
best_metric_score = float("-inf")
best_threshold = 0.5
best_epoch = 0
epochs_without_improvement = 0
start_epoch = 1

if mode == "resume":
    if not LAST_CHECKPOINT.is_file():
        raise FileNotFoundError(
            f"Resume requested but last.ckpt missing: "
            f"{LAST_CHECKPOINT}"
        )

    resume_state = load_checkpoint(
        LAST_CHECKPOINT,
        DEVICE,
    )

    # Hard guard: same experiment identity/config essentials.
    old_config = resume_state["config"]

    identity_fields = [
        ("experiment", "region"),
        ("experiment", "model_name"),
        ("experiment", "seed"),
        ("data", "image_size"),
        ("data", "relative_path_column"),
    ]

    mismatches = []
    for section, key in identity_fields:
        old_value = old_config[section][key]
        new_value = CONFIG[section][key]
        if old_value != new_value:
            mismatches.append(
                f"{section}.{key}: "
                f"checkpoint={old_value!r}, current={new_value!r}"
            )

    if mismatches:
        raise ValueError(
            "Resume config identity mismatch:\n"
            + "\n".join(mismatches)
        )

    model.load_state_dict(
        resume_state["model_state_dict"]
    )
    optimizer.load_state_dict(
        resume_state["optimizer_state_dict"]
    )
    scheduler.load_state_dict(
        resume_state["scheduler_state_dict"]
    )
    scaler.load_state_dict(
        resume_state["scaler_state_dict"]
    )

    restore_rng_state(
        resume_state["rng_state"]
    )
    loader_generator.set_state(
        resume_state["loader_generator_state"]
    )

    history = list(
        resume_state["history"]
    )
    best_metric_score = float(
        resume_state["best_metric_score"]
    )
    best_threshold = float(
        resume_state["best_threshold"]
    )
    best_epoch = int(
        resume_state["best_epoch"]
    )
    epochs_without_improvement = int(
        resume_state["epochs_without_improvement"]
    )

    start_epoch = int(
        resume_state["epoch"]
    ) + 1


    print(
        f"RESUME PASSED — continuing from epoch "
        f"{start_epoch}"
    )
    print("Best metric:", best_metric_score)
    print("Best threshold:", best_threshold)

else:
    print("NEW RUN — starting at epoch 1")


In [ ]:
# ============================================================
# 14. TRAINING LOOP
# ============================================================
import json

import numpy as np
import pandas as pd

EPOCHS = int(T["epochs"])
PATIENCE = int(T["early_stopping_patience"])
MONITOR = str(T["monitor_metric"])
FALLBACK = str(T["fallback_metric"])
AMP_OVERFLOW_ABORT_AFTER = int(
    T.get("amp_overflow_abort_after", 8)
)

if start_epoch > EPOCHS:
    print(
        "Training already reached configured epoch count. "
        "Skipping training loop."
    )

for epoch in range(start_epoch, EPOCHS + 1):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 80)

    train_out = run_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        device=DEVICE,
        amp_enabled=AMP_ENABLED,
        scaler=scaler,
        gradient_clip_norm=float(
            T["gradient_clip_norm"]
        ),
        amp_overflow_abort_after=AMP_OVERFLOW_ABORT_AFTER,
        optimizer=optimizer,
    )

    train_metrics = compute_metrics(
        train_out["labels"],
        train_out["probabilities"],
        threshold=float(T["default_threshold"]),
    )

    val_out = run_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        amp_enabled=AMP_ENABLED,
        scaler=scaler,
        gradient_clip_norm=float(
            T["gradient_clip_norm"]
        ),
        amp_overflow_abort_after=AMP_OVERFLOW_ABORT_AFTER,
        optimizer=None,
    )

    epoch_threshold, _ = (
        select_best_f1_threshold(
            val_out["labels"],
            val_out["probabilities"],
            threshold_min=float(T["threshold_search_min"]),
            threshold_max=float(T["threshold_search_max"]),
            threshold_steps=int(T["threshold_search_steps"]),
            default_threshold=float(T["default_threshold"]),
        )
    )

    val_metrics = compute_metrics(
        val_out["labels"],
        val_out["probabilities"],
        threshold=epoch_threshold,
    )

    score = val_metrics.get(
        MONITOR,
        float("nan"),
    )

    if not np.isfinite(score):
        score = val_metrics.get(
            FALLBACK,
            float("nan"),
        )

    if not np.isfinite(score):
        raise FloatingPointError(
            "Primary and fallback validation metric "
            "are non-finite."
        )

    row = {
        "epoch": epoch,
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"],
        "train_loss": train_out["loss"],
        "val_loss": val_out["loss"],
        "val_threshold": epoch_threshold,
        "train_amp_overflow_steps": int(
            train_out.get("amp_overflow_steps", 0)
        ),
        **{
            f"train_{key}": value
            for key, value in train_metrics.items()
        },
        **{
            f"val_{key}": value
            for key, value in val_metrics.items()
        },
    }

    history.append(row)

    improved = (
        score >
        best_metric_score + float(T["min_improvement"])
    )

    if improved:
        best_metric_score = float(score)
        best_threshold = float(epoch_threshold)
        best_epoch = int(epoch)
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # Advance scheduler before saving end-of-epoch state.
    scheduler.step()

    end_of_epoch_state = build_checkpoint_state(
        epoch=epoch,
        best_metric_score=best_metric_score,
        best_threshold=best_threshold,
        history=history,
        best_epoch=best_epoch,
        epochs_without_improvement=epochs_without_improvement,
    )

    # last.ckpt always represents the complete state for epoch+1 resume.
    atomic_save_checkpoint(
        end_of_epoch_state,
        LAST_CHECKPOINT,
    )

    # best.ckpt is saved from the same complete end-of-epoch state.
    if improved:
        atomic_save_checkpoint(
            end_of_epoch_state,
            BEST_CHECKPOINT,
        )
        print("New BEST checkpoint saved.")

    epoch_path = (
        CHECKPOINT_DIR /
        f"epoch_{epoch:03d}.ckpt"
    )

    atomic_save_checkpoint(
        end_of_epoch_state,
        epoch_path,
    )

    # Only after checkpoint success do we persist human-readable history.
    pd.DataFrame(history).to_csv(
        METRICS_DIR / "training_history.csv",
        index=False,
    )

    epoch_files = sorted(
        CHECKPOINT_DIR.glob(
            "epoch_*.ckpt"
        )
    )

    keep_n = int(
        T["keep_last_n_epoch_checkpoints"]
    )

    while len(epoch_files) > keep_n:
        epoch_files.pop(0).unlink()

    print(json.dumps(row, indent=2))
    print("Best score:", best_metric_score)
    print("Best epoch:", best_epoch)
    print("Threshold:", best_threshold)
    print(
        "Epochs without improvement:",
        epochs_without_improvement,
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
# ============================================================
# 15. FRESH BEST.CKPT INFERENCE GATE + FINAL TEST
# ============================================================

import json
import torch

if not BEST_CHECKPOINT.is_file():
    raise RuntimeError(
        "best.ckpt was not created."
    )

best_state = load_checkpoint(
    BEST_CHECKPOINT,
    DEVICE,
)

inference_model = SwinV2TinyBinaryClassifier(
    pretrained=False,
    dropout=float(CONFIG["model"]["dropout"]),
).to(DEVICE)

inference_model.load_state_dict(
    best_state["model_state_dict"]
)

inference_model.eval()

fresh_batch = next(iter(test_loader))

with torch.no_grad():
    fresh_logits = inference_model(
        fresh_batch["image"].to(DEVICE)
    )

assert (
    fresh_logits.shape[0]
    ==
    fresh_batch["image"].shape[0]
)

assert_finite_tensor(
    fresh_logits,
    "fresh best.ckpt inference",
)

print("Fresh best.ckpt inference: PASSED")

FINAL_THRESHOLD = float(
    best_state["best_threshold"]
)

# Test is used for final evaluation only.
test_out = run_epoch(
    model=inference_model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    amp_enabled=AMP_ENABLED,
    scaler=scaler,
    gradient_clip_norm=float(
        T["gradient_clip_norm"]
    ),
    amp_overflow_abort_after=int(
        T.get("amp_overflow_abort_after", 8)
    ),
    optimizer=None,
)

test_metrics = compute_metrics(
    test_out["labels"],
    test_out["probabilities"],
    threshold=FINAL_THRESHOLD,
)

test_metrics.update(
    {
        "threshold": FINAL_THRESHOLD,
        "best_epoch": int(best_state["epoch"]),
        "run_id": run_id,
    }
)

save_json_atomic(
    test_metrics,
    METRICS_DIR / "test_frame_metrics.json",
)

print(json.dumps(test_metrics, indent=2))


In [ ]:
# ============================================================
# 16. FRAME-LEVEL + VIDEO-LEVEL PREDICTIONS
# ============================================================

import json
import pandas as pd

frame_predictions = (
    test_out["probabilities"]
    >= FINAL_THRESHOLD
).astype(np.int64)

predictions_df = pd.DataFrame(
    {
        "sample_id": test_out["sample_ids"],
        "video_id": test_out["video_ids"],
        "image_path": test_out["paths"],
        "true_label": test_out["labels"],
        "fake_probability": test_out["probabilities"],
        "predicted_label": frame_predictions,
        "threshold": FINAL_THRESHOLD,
    }
)

predictions_df["true_class"] = (
    predictions_df["true_label"]
    .map({0: "real", 1: "fake"})
)

predictions_df["predicted_class"] = (
    predictions_df["predicted_label"]
    .map({0: "real", 1: "fake"})
)

predictions_df.to_csv(
    PREDICTIONS_DIR / "test_frame_predictions.csv",
    index=False,
)


if video_identity_reliable:
    video_predictions_df = (
        predictions_df
        .groupby("video_id", as_index=False)
        .agg(
            true_label=("true_label", "first"),
            fake_probability=("fake_probability", "mean"),
            frame_count=("sample_id", "count"),
        )
    )

    video_predictions_df["predicted_label"] = (
        video_predictions_df["fake_probability"]
        >= FINAL_THRESHOLD
    ).astype(np.int64)

    video_metrics = compute_metrics(
        video_predictions_df["true_label"].to_numpy(),
        video_predictions_df[
            "fake_probability"
        ].to_numpy(),
        threshold=FINAL_THRESHOLD,
    )

    video_metrics.update(
        {
            "available": True,
            "threshold": FINAL_THRESHOLD,
            "run_id": run_id,
        }
    )

    video_predictions_df.to_csv(
        PREDICTIONS_DIR / "test_video_predictions.csv",
        index=False,
    )
else:
    # Do not publish misleading "video-level" metrics when metadata does not
    # contain trustworthy source-video identities.
    video_metrics = {
        "available": False,
        "run_id": run_id,
        "reason": (
            "video_id appears to be a label/split bucket rather than a true "
            "source-video identifier; per-original-video metrics are disabled."
        ),
    }

    pd.DataFrame(
        columns=[
            "video_id",
            "true_label",
            "fake_probability",
            "frame_count",
            "predicted_label",
        ]
    ).to_csv(
        PREDICTIONS_DIR / "test_video_predictions.csv",
        index=False,
    )

save_json_atomic(
    video_metrics,
    METRICS_DIR / "test_video_metrics.json",
)

print("VIDEO LEVEL")
print(json.dumps(video_metrics, indent=2))


In [ ]:
# ============================================================
# 17. FIGURES
# ============================================================

import pandas as pd

history_df = pd.DataFrame(
    best_state.get("history", history)
)

if history_df.empty:
    raise RuntimeError(
        "Training history is empty."
    )

generate_all_figures(
    history_df=history_df,
    labels=test_out["labels"],
    probabilities=test_out["probabilities"],
    threshold=FINAL_THRESHOLD,
    figures_dir=FIGURES_DIR,
    min_short_edge=int(
        CONFIG["quality"]["min_figure_short_edge_px"]
    ),
)

print("Figures:", FIGURES_DIR)


In [ ]:
# ============================================================
# 18. FINAL QUALITY GATES + RUN SUMMARY + OUTPUT MANIFEST
# ============================================================
import json
import pandas as pd
from PIL import Image

run_summary = {
    "run_id": run_id,
    "architecture": "Swin V2 Tiny",
    "region": "Eyebrow ROI",
    "source_data": "Deneyler/Nazlıcan/Deney 1/Kaş",
    "mode": mode,
    "seed": SEED,
    "pretrained": bool(CONFIG["model"]["pretrained"]),
    "normalization": normalization_audit,
    "best_epoch": int(best_state["epoch"]),
    "best_validation_score": float(best_state["best_metric_score"]),
    "validation_selected_threshold": FINAL_THRESHOLD,
    "test_frame_metrics": test_metrics,
    "test_video_metrics": video_metrics,
    "train_frames": len(train_df),
    "val_frames": len(val_df),
    "test_frames": len(test_df),
    "train_videos": int(train_df[video_col].nunique()),
    "val_videos": int(val_df[video_col].nunique()),
    "test_videos": int(test_df[video_col].nunique()),
    "quality_gates": {
        "metadata_schema": "passed",
        "accounting_equality": "passed",
        "source_video_split_isolation": "passed",
        "two_batch_smoke_test": "passed",
        "checkpoint_continuity": "passed",
        "numerical_nan_inf": "passed",
        "fresh_best_checkpoint_inference": "passed",
        "test_used_only_for_final_evaluation": True,
        "figure_language": "English",
        "minimum_figure_short_edge_px": int(CONFIG["quality"]["min_figure_short_edge_px"]),
    },
}
save_json_atomic(run_summary, RUN_DIR / "run_summary.json")

required_outputs = [
    RUN_DIR / "config_resolved.yaml",
    RUN_DIR / "requirements_lock.txt",
    RUN_DIR / "environment.json",
    RUN_DIR / "run_summary.json",
    LOG_DIR / "pipeline.log",
    ARTIFACTS_DIR / "metadata_schema_audit.json",
    ARTIFACTS_DIR / "data_accounting.json",
    ARTIFACTS_DIR / "split_leakage_report.json",
    ARTIFACTS_DIR / "metadata_used.csv",
    ARTIFACTS_DIR / "dataset_summary.csv",
    ARTIFACTS_DIR / "normalization_audit.json",
    ARTIFACTS_DIR / "input_numerical_sanity.json",
    ARTIFACTS_DIR / "checkpoint_continuity_test.json",
    CHECKPOINT_DIR / "last.ckpt",
    CHECKPOINT_DIR / "best.ckpt",
    METRICS_DIR / "training_history.csv",
    METRICS_DIR / "test_frame_metrics.json",
    METRICS_DIR / "test_video_metrics.json",
    PREDICTIONS_DIR / "test_frame_predictions.csv",
    PREDICTIONS_DIR / "test_video_predictions.csv",
    FIGURES_DIR / "training_validation_loss.png",
    FIGURES_DIR / "training_validation_accuracy.png",
    FIGURES_DIR / "validation_roc_auc.png",
    FIGURES_DIR / "test_confusion_matrix.png",
    FIGURES_DIR / "test_roc_curve.png",
    FIGURES_DIR / "test_precision_recall_curve.png",
]

missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise RuntimeError("Final quality gate FAILED. Missing:\n" + "\n".join(missing))

# Leakage re-check immediately before delivery.
assert train_videos.isdisjoint(val_videos)
assert train_videos.isdisjoint(test_videos)
assert val_videos.isdisjoint(test_videos)

# English figure generation is implemented in evaluation/plots.py; resolution is checked again here.
for figure_path in FIGURES_DIR.glob("*.png"):
    with Image.open(figure_path) as image:
        assert min(image.size) >= int(CONFIG["quality"]["min_figure_short_edge_px"]), (
            f"Figure quality failed: {figure_path} -> {image.size}"
        )

# Fresh best checkpoint can still be loaded after all outputs have been produced.
final_check = load_checkpoint(BEST_CHECKPOINT, DEVICE)
assert int(final_check["epoch"]) == int(best_state["epoch"])

# Final log lines are written and flushed BEFORE hashing outputs.
logger.info("ALL FINAL QUALITY GATES PASSED")
logger.info("Run directory: %s", RUN_DIR)
for handler in logger.handlers:
    handler.flush()

# Build a human-verifiable manifest of every output except the manifest itself.
# No run output file is modified after this block, so recorded hashes stay valid.
manifest_rows = []
for path in sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "output_manifest.csv"):
    relative = path.relative_to(RUN_DIR)
    top = relative.parts[0] if len(relative.parts) > 1 else "root"
    manifest_rows.append({
        "relative_path": relative.as_posix(),
        "category": top,
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(RUN_DIR / "output_manifest.csv", index=False)
if manifest_df.empty:
    raise RuntimeError("Output manifest is empty.")

print("=" * 80)
print("ALL FINAL QUALITY GATES PASSED")
print("=" * 80)
print("RUN DIRECTORY:")
print(RUN_DIR)
print("\nFRAME LEVEL:")
print(json.dumps(test_metrics, indent=2))
print("\nVIDEO LEVEL:")
print(json.dumps(video_metrics, indent=2))
print("\nOUTPUT STRUCTURE:")
for name in [
    "checkpoints", "logs", "metrics", "predictions", "figures", "artifacts",
    "config_resolved.yaml", "requirements_lock.txt", "environment.json",
    "output_manifest.csv", "run_summary.json",
]:
    print(" -", name)


## Resume nasıl kullanılır?

Colab veya GPU oturumu kesilirse **aynı run klasöründen devam etmek** için Config hücresinde yalnızca:

```python
"mode": "resume",
"resume_run_id": "20260807_...._eye_swinv2_tiny_seed42",
```

olarak değiştirin ve notebook'u baştan çalıştırın.

Resume aşaması `last.ckpt` içinden:
- model,
- optimizer,
- scheduler,
- AMP scaler,
- Python / NumPy / Torch / CUDA RNG,
- DataLoader generator state,
- history,
- best metric,
- best epoch,
- early-stopping counter,
- validation threshold

bilgilerini geri yükler ve `epoch + 1`'den devam eder.